<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/pinn_deepxde2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install deepxde

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [deepxde]


In [2]:
import os
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde
import numpy as np
import torch
import plotly.express as px
import itertools

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


In [3]:
dde.config.set_default_float("float64")

Set the default float type to float64


In [4]:
kappa = 1.0
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

In [5]:
def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

In [6]:
def exact(x):
    xx = x[:, 0:1]
    tt = x[:, 1:2]
    return np.exp(-tt) * np.sin(xx)

In [7]:
iterations = 10000
epsilon = 1e-4

In [8]:
def output(x, y):
    x_col = x[:, 0:1]
    t_col = x[:, 1:2]
    return torch.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

In [9]:
XT_test = geomtime.uniform_points(5000)
y_true = exact(XT_test)

In [10]:
fig = px.scatter(
    x=XT_test[:, 0],
    y=XT_test[:, 1],
    labels={"x": "Координата x", "y": "Время t"}
)

fig.add_shape(
    type="rect",
    x0=XT_test[:, 0].min(), y0=XT_test[:, 1].min(),
    x1=XT_test[:, 0].max(), y1=XT_test[:, 1].max(),
    line=dict(color="blue", width=2),
    fillcolor="rgba(0, 0, 255, 0.05)",
)

fig.update_layout(template="plotly_white")
fig.update_traces(marker=dict(size=6, color="red"))
fig.show()

In [17]:
search_space = {"layers": list(range(2, 8, 2)), "neurons": list(range(20, 60, 10)), "num_domain": list(range(100, 5000, 200))}
keys = search_space.keys()
combinations = itertools.product(*search_space.values())
print(len(list(combinations)))

300


In [12]:
dde.optimizers.config.set_LBFGS_options(maxiter=5000)
for values in combinations:
  params = dict(zip(keys, values))

  dde.config.set_random_seed(42)

  data = dde.data.TimePDE(geomtime, pde, [], num_domain=params["num_domain"], num_test=5000, solution=exact, train_distribution="uniform")

  net = dde.nn.FNN([2] + [params["neurons"]] * params["layers"] + [1], "tanh", "Glorot normal")

  net.apply_output_transform(output)

  model = dde.Model(data, net)

  print("Стартовые веса: ", model.net.linears[0].weight.detach()[0, :3].cpu().numpy())
  print("Стартовые смещения: ", model.net.linears[0].bias.detach()[:3].cpu().numpy())

  model.compile("adam", lr=1e-3, metrics=["l2 relative error"])
  model.train(iterations=iterations)

  model.compile("L-BFGS", metrics=["l2 relative error"])
  model.train()

  y_pred = model.predict(XT_test)
  l2_error = dde.metrics.l2_relative_error(y_true, y_pred)

  print(f"{params} | L2 Error: {l2_error:.4e}")

  if l2_error < epsilon:
    break

Стартовые веса:  [-0.01924063  0.2973672 ]
Стартовые смещения:  [0. 0. 0.]
Compiling model...
'compile' took 6.650551 s

Training model...

Step      Train loss    Test loss     Test metric   
0         [4.81e+00]    [3.93e+00]    [1.47e+00]    
1000      [7.79e-04]    [1.08e-03]    [1.31e-02]    
2000      [2.55e-04]    [4.12e-04]    [8.34e-03]    
3000      [9.28e-05]    [1.70e-04]    [5.10e-03]    
4000      [3.38e-05]    [7.10e-05]    [3.03e-03]    
5000      [1.69e-05]    [3.64e-05]    [2.04e-03]    
6000      [1.17e-05]    [2.62e-05]    [1.79e-03]    
7000      [8.94e-06]    [2.22e-05]    [1.75e-03]    
8000      [7.56e-06]    [2.01e-05]    [1.70e-03]    
9000      [6.64e-06]    [1.84e-05]    [1.65e-03]    
10000     [5.97e-06]    [1.71e-05]    [1.60e-03]    

Best model at step 10000:
  train loss: 5.97e-06
  test loss: 1.71e-05
  test metric: [1.60e-03]

'train' took 25.838514 s

Compiling model...
'compile' took 0.000703 s

Training model...

Step      Train loss    Test loss 